# Lesson 7 · Model Optimization II — Memory & Attention
### REAL AWS Implementation: vLLM on a SageMaker GPU Endpoint

**Module 05 · AI Platform Engineering · BITS Pilani**

Deploys **vLLM as a real, managed SageMaker endpoint** via the AWS **Large Model Inference (LMI)** container.
No simulation, no local fallback — every block calls a real AWS service. The vLLM engine (PagedAttention +
continuous batching + Flash Attention) runs on a real GPU instance and is benchmarked over the network.

## Key hardening: automatic GPU instance selection

GPU endpoint quota varies per instance type and region. This notebook **auto-sweeps** a list of GPU
instances and deploys on the first one your account has quota for — so a zero quota on `ml.g5.xlarge`
falls through to `ml.g4dn.xlarge` automatically instead of failing. Cost metrics adjust to the chosen instance.

## What runs on AWS (all real)

| Step | AWS service |
|------|-------------|
| 0–1 | STS identity, region, config |
| 2 | IAM — self-healing / clean execution role |
| 3 | ECR — LMI (vLLM) container image |
| 4 | SageMaker CreateModel — vLLM engine config |
| 5 | SageMaker Endpoint — **auto-sweeps GPU instances**, deploys on the first with quota |
| 6 | Poll to InService |
| 7 | SageMaker Runtime — real smoke-test inference |
| 8 | Concurrency sweep — real continuous batching |
| 9 | Four metrics (TTFT/TPOT/throughput/p99) + cost/1M tokens |
| 10 | Guarded teardown |

## Prerequisites

- Quota ≥ 1 on **at least one** of: `ml.g4dn.xlarge`, `ml.g5.xlarge`, `ml.g4dn.2xlarge`, `ml.g5.2xlarge`
  (endpoint usage). The sweep finds whichever you have.
- Execution role with SageMaker + S3 + ECR access. Block 2 builds a clean one if the default fails.

## Cost + time

First deploy ~8–12 min. Endpoint bills continuously (~$0.74/hr g4dn, ~$1.40/hr g5). **Run Block 10 when done.**


## 1. Preflight

In [1]:
# Block 0 - Preflight (read-only)
import os, sys, json, time, importlib
import boto3
from botocore.exceptions import ClientError

_ok = True
def _pkg(n):
    global _ok
    try:
        m = importlib.import_module(n); print(f"  PASS  {n:12s} {getattr(m,'__version__','?')}")
    except ImportError:
        print(f"  FAIL  {n} missing"); _ok = False

print("=== Packages ==="); _pkg("boto3"); _pkg("sagemaker")
print("\n=== Identity ===")
sts = boto3.client("sts"); ident = sts.get_caller_identity()
print(f"  Account: {ident['Account']}")
print(f"  Caller:  {ident['Arn']}")
_region = boto3.Session().region_name or os.environ.get("AWS_DEFAULT_REGION", "us-east-1")
print(f"  Region:  {_region}")
if not _ok: raise RuntimeError("Preflight failed.")
print("\nPreflight passed.")


=== Packages ===
  PASS  boto3        1.43.56
  FAIL  sagemaker missing

=== Identity ===


NoCredentialsError: Unable to locate credentials

## 2. Configuration

`GPU_INSTANCE_CANDIDATES` is the sweep order — the endpoint deploys on the first one with quota.

In [2]:
# Block 1 - Config
AWS_REGION_NAME     = "us-east-1"        # region where you have GPU quota
ALLOW_AWS_MUTATIONS = True

PROJECT_NAME  = "lesson7-vllm"
ENVIRONMENT   = "demo"
HF_MODEL_ID   = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"   # small, loads fast on any GPU
MODEL_S3_URI  = None                     # or "s3://bucket/prefix/" to serve from S3
NUM_GPUS      = 1

# The endpoint auto-sweeps these in order and uses the first with quota.
# (Learned from the live run: g5 quota was 0, g4dn worked — so g4dn is first.)
GPU_INSTANCE_CANDIDATES = [
    "ml.g4dn.xlarge",    # T4 16GB  ~$0.74/hr
    "ml.g5.xlarge",      # A10G 24GB ~$1.41/hr
    "ml.g4dn.2xlarge",   # T4 16GB  ~$0.94/hr
    "ml.g5.2xlarge",     # A10G 24GB ~$1.52/hr
    "ml.p3.2xlarge",     # V100 16GB ~$3.83/hr
]
# Approx USD/hr per instance for the cost metric (verify for your region).
INSTANCE_HOURLY = {
    "ml.g4dn.xlarge": 0.736, "ml.g5.xlarge": 1.408, "ml.g4dn.2xlarge": 0.94,
    "ml.g5.2xlarge": 1.515, "ml.p3.2xlarge": 3.825,
}

os.environ["AWS_DEFAULT_REGION"] = AWS_REGION_NAME
os.environ["AWS_REGION"]         = AWS_REGION_NAME
region       = AWS_REGION_NAME
boto_session = boto3.Session(region_name=region)
s3  = boto_session.client("s3");   sm  = boto_session.client("sagemaker")
smr = boto_session.client("sagemaker-runtime"); iam = boto_session.client("iam")
cw  = boto_session.client("cloudwatch")
_ident = sts.get_caller_identity(); account_id = _ident["Account"]; caller_arn = _ident["Arn"]

import datetime as dt
timestamp       = dt.datetime.utcnow().strftime("%Y%m%d-%H%M%S")
model_name      = f"{PROJECT_NAME}-{ENVIRONMENT}-model-{timestamp}"
endpoint_name   = f"{PROJECT_NAME}-{ENVIRONMENT}-ep"
endpoint_config = None   # set by the sweep in Block 5
chosen_instance = None   # set by the sweep in Block 5

print(f"Region:   {region}")
print(f"Account:  {account_id}")
print(f"Model:    {HF_MODEL_ID}")
print(f"Sweep:    {GPU_INSTANCE_CANDIDATES}")
print(f"Endpoint: {endpoint_name}")


Region:   us-east-1
Account:  797715838180
Model:    TinyLlama/TinyLlama-1.1B-Chat-v1.0
Sweep:    ['ml.g4dn.xlarge', 'ml.g5.xlarge', 'ml.g4dn.2xlarge', 'ml.g5.2xlarge', 'ml.p3.2xlarge']
Endpoint: lesson7-vllm-demo-ep


## 3. Execution role — self-healing / clean role

In [3]:
# Block 2 - Resolve a working execution role
def _default_role():
    try:
        from sagemaker import get_execution_role
        return get_execution_role()
    except Exception:
        if ":assumed-role/" in caller_arn:
            return f"arn:aws:iam::{account_id}:role/{caller_arn.split('/')[1]}"
        return caller_arn

execution_role = _default_role()
print(f"Default execution role: {execution_role}")

def build_clean_role():
    role_name = "Lesson7VLLMExecRole"
    trust = {"Version": "2012-10-17", "Statement": [{
        "Effect": "Allow", "Principal": {"Service": "sagemaker.amazonaws.com"},
        "Action": ["sts:AssumeRole","sts:TagSession","sts:SetContext","sts:SetSourceIdentity"]}]}
    try:
        iam.create_role(RoleName=role_name, AssumeRolePolicyDocument=json.dumps(trust),
                        Description="Clean role for Lesson 7 vLLM endpoint")
        print(f"Created {role_name}")
    except iam.exceptions.EntityAlreadyExistsException:
        iam.update_assume_role_policy(RoleName=role_name, PolicyDocument=json.dumps(trust))
        print(f"Reset trust on {role_name}")
    for arn in ["arn:aws:iam::aws:policy/AmazonSageMakerFullAccess",
                "arn:aws:iam::aws:policy/AmazonS3FullAccess",
                "arn:aws:iam::aws:policy/AmazonEC2ContainerRegistryReadOnly"]:
        try: iam.attach_role_policy(RoleName=role_name, PolicyArn=arn)
        except Exception: pass
    print("Waiting 60s for IAM propagation..."); time.sleep(60)
    return f"arn:aws:iam::{account_id}:role/{role_name}"

# Set True if the default role fails at CreateModel/endpoint with a trust/S3 error.
USE_CLEAN_ROLE = False
if USE_CLEAN_ROLE and ALLOW_AWS_MUTATIONS:
    execution_role = build_clean_role()
    print(f"Using clean role: {execution_role}")


sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3Bucket
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3ObjectKeyPrefix
Default execution role: arn:aws:iam::797715838180:role/service-role/AmazonSageMakerAdminIAMExecutionRole_1


## 4. Resolve the LMI (vLLM) container image

In [4]:
# Block 3 - LMI container image URI (newest-first + hardcoded fallback)
lmi_image = None
try:
    import sagemaker.image_uris as _img_uris
    for _ver in ["0.31.0", "0.30.0", "0.29.0"]:
        try:
            lmi_image = _img_uris.retrieve(framework="djl-lmi", region=region, version=_ver)
            print(f"Resolved LMI image via SDK (v{_ver}): {lmi_image}")
            break
        except Exception:
            continue
except Exception as e:
    print(f"SDK lookup unavailable ({type(e).__name__}); using hardcoded fallback.")

if lmi_image is None:
    # Confirmed-real published tag; DLC account 763104351884 in most regions.
    lmi_image = f"763104351884.dkr.ecr.{region}.amazonaws.com/djl-inference:0.31.0-lmi13.0.0-cu124"
    print(f"Using hardcoded LMI image: {lmi_image}")
print(f"\nLMI image: {lmi_image}")


Resolved LMI image via SDK (v0.31.0): 763104351884.dkr.ecr.us-east-1.amazonaws.com/djl-inference:0.31.0-lmi13.0.0-cu124

LMI image: 763104351884.dkr.ecr.us-east-1.amazonaws.com/djl-inference:0.31.0-lmi13.0.0-cu124


## 5. Create the SageMaker Model (vLLM engine config)

In [5]:
# Block 4 - CreateModel with vLLM engine
vllm_env = {
    "HF_MODEL_ID":                   HF_MODEL_ID,
    "OPTION_ROLLING_BATCH":          "vllm",          # vLLM engine (continuous batching)
    "OPTION_MAX_ROLLING_BATCH_SIZE": "16",
    "OPTION_TENSOR_PARALLEL_DEGREE": str(NUM_GPUS),
    "OPTION_DTYPE":                  "fp16",          # T4 (g4dn) and A10G (g5) both support fp16
    "OPTION_MAX_MODEL_LEN":          "2048",
    "OPTION_TRUST_REMOTE_CODE":      "true",
}
_primary = {"Image": lmi_image, "Environment": vllm_env}
if MODEL_S3_URI:
    _primary["ModelDataSource"] = {"S3DataSource": {
        "S3Uri": MODEL_S3_URI, "S3DataType": "S3Prefix", "CompressionType": "None"}}
    vllm_env.pop("HF_MODEL_ID", None)
    vllm_env["OPTION_MODEL_ID"] = "/opt/ml/model"

if ALLOW_AWS_MUTATIONS:
    try:
        sm.create_model(ModelName=model_name, PrimaryContainer=_primary, ExecutionRoleArn=execution_role)
        print(f"Created Model: {model_name}  (engine: vLLM)")
    except ClientError as e:
        if "already exists" in str(e).lower():
            print(f"Model exists: {model_name}")
        else:
            print(f"CreateModel failed: {e}")
            print("If role/trust error: set USE_CLEAN_ROLE=True in Block 2, re-run Blocks 2 and 4.")
            raise
else:
    print("Skipped.")


Created Model: lesson7-vllm-demo-model-20260816-112833  (engine: vLLM)


## 6. Deploy — auto-sweep GPU instances

Tries each candidate instance in order; deploys on the first your account has quota for. This is the
fix for the `ResourceLimitExceeded` you hit — instead of failing on one instance, it finds one that works.

In [6]:
# Block 5 - Create endpoint, auto-sweeping instance types for quota
def _cleanup_config(cfg):
    try: sm.delete_endpoint_config(EndpointConfigName=cfg)
    except Exception: pass

if ALLOW_AWS_MUTATIONS:
    chosen_instance = None
    for inst in GPU_INSTANCE_CANDIDATES:
        ts  = dt.datetime.utcnow().strftime("%Y%m%d-%H%M%S")
        cfg = f"{PROJECT_NAME}-{ENVIRONMENT}-cfg-{inst.replace('.','-')}-{ts}"
        try:
            sm.create_endpoint_config(
                EndpointConfigName=cfg,
                ProductionVariants=[{
                    "VariantName": "AllTraffic", "ModelName": model_name,
                    "InstanceType": inst, "InitialInstanceCount": 1,
                    "ContainerStartupHealthCheckTimeoutInSeconds": 900,
                }],
            )
        except ClientError as e:
            print(f"  {inst}: config error {e.response['Error']['Code']}")
            continue
        # Try to attach an endpoint - this is where quota is enforced
        try:
            try:
                sm.create_endpoint(EndpointName=endpoint_name, EndpointConfigName=cfg)
            except ClientError as e:
                if "already exist" in str(e).lower():
                    sm.update_endpoint(EndpointName=endpoint_name, EndpointConfigName=cfg)
                else:
                    raise
            print(f"  {inst}: ACCEPTED - endpoint creating")
            chosen_instance = inst
            endpoint_config = cfg
            break
        except ClientError as e:
            code = e.response["Error"]["Code"]
            print(f"  {inst}: {code}")
            _cleanup_config(cfg)   # remove the orphaned config and try next instance

    if chosen_instance is None:
        raise RuntimeError(
            "No GPU instance in GPU_INSTANCE_CANDIDATES has quota in this region.\n"
            "Request an increase (Service Quotas > SageMaker > '<instance> for endpoint usage'), "
            "or add more instance types / try another region."
        )
    print(f"\nDeploying on: {chosen_instance}")
    print(f"Config:       {endpoint_config}")
else:
    print("Skipped.")


  ml.g4dn.xlarge: ACCEPTED - endpoint creating

Deploying on: ml.g4dn.xlarge
Config:       lesson7-vllm-demo-cfg-ml-g4dn-xlarge-20260816-112835


In [7]:
# Block 6 - Poll until InService (~8-12 min first deploy)
if ALLOW_AWS_MUTATIONS:
    print(f"Polling {endpoint_name} on {chosen_instance} ...")
    while True:
        info = sm.describe_endpoint(EndpointName=endpoint_name)
        status = info["EndpointStatus"]
        print(f"  {dt.datetime.utcnow().isoformat(timespec='seconds')}  {status}")
        if status == "InService":
            print("\nInService - vLLM is live on a real GPU.")
            break
        if status == "Failed":
            print("\nFailureReason:", info.get("FailureReason", "(none)"))
            print("If CUDA/dtype error on T4: keep OPTION_DTYPE=fp16 (already set).")
            print("If model too big: use a smaller HF_MODEL_ID or a larger instance.")
            raise RuntimeError("Endpoint failed - see FailureReason.")
        time.sleep(30)
else:
    print("Skipped.")


Polling lesson7-vllm-demo-ep on ml.g4dn.xlarge ...
  2026-08-16T11:28:36  Creating
  2026-08-16T11:29:06  Creating
  2026-08-16T11:29:36  Creating
  2026-08-16T11:30:06  Creating
  2026-08-16T11:30:36  Creating
  2026-08-16T11:31:06  Creating
  2026-08-16T11:31:36  Creating
  2026-08-16T11:32:06  Creating
  2026-08-16T11:32:36  Creating
  2026-08-16T11:33:06  Creating
  2026-08-16T11:33:37  Creating
  2026-08-16T11:34:07  Creating
  2026-08-16T11:34:37  Creating
  2026-08-16T11:35:07  Creating
  2026-08-16T11:35:37  Creating
  2026-08-16T11:36:07  InService

InService - vLLM is live on a real GPU.


## 7. Smoke test — one real inference

In [8]:
# Block 7 - Single real inference through vLLM
def invoke_vllm(prompt, max_new_tokens=64, stream=False, **params):
    payload = {"inputs": prompt,
               "parameters": {"max_new_tokens": max_new_tokens, "temperature": 0.7, **params}}
    if not stream:
        resp = smr.invoke_endpoint(EndpointName=endpoint_name,
                                   ContentType="application/json", Body=json.dumps(payload))
        body = json.loads(resp["Body"].read())
        if isinstance(body, dict):  return body.get("generated_text", body)
        if isinstance(body, list) and body: return body[0].get("generated_text", body[0])
        return body
    resp = smr.invoke_endpoint_with_response_stream(
        EndpointName=endpoint_name, ContentType="application/json",
        Body=json.dumps({**payload, "stream": True}))
    return resp["Body"]

if ALLOW_AWS_MUTATIONS:
    print("Smoke test - one real inference...\n")
    out = invoke_vllm("Explain PagedAttention in one sentence:", max_new_tokens=48)
    print("Response:", out)
    print("\nvLLM endpoint answering. Safe to benchmark.")
else:
    print("Skipped.")


Smoke test - one real inference...

Response:  PagedAttention is a technique that allows us to efficiently compute the attention scores for each token in a sequence.

vLLM endpoint answering. Safe to benchmark.


## 8. Concurrency sweep — continuous batching under real load

In [9]:
# Block 8 - Concurrency sweep against the real endpoint
import concurrent.futures, numpy as np, pandas as pd

def _one(prompt="Summarise the benefit of continuous batching:"):
    t0 = time.perf_counter(); invoke_vllm(prompt, max_new_tokens=32); return time.perf_counter()-t0

if ALLOW_AWS_MUTATIONS:
    _one()  # warm up
    rows = []
    for c in [1, 2, 4, 8, 16]:
        t0 = time.perf_counter()
        with concurrent.futures.ThreadPoolExecutor(max_workers=c) as ex:
            list(ex.map(lambda _: _one(), range(c)))
        wall = time.perf_counter() - t0
        rows.append({"concurrency": c, "wall_s": round(wall,2), "req_per_s": round(c/wall,2)})
        print(f"  concurrency={c:2d}  wall={wall:5.2f}s  {c/wall:5.2f} req/s")
    df_sweep = pd.DataFrame(rows)
    print("\n" + df_sweep.to_string(index=False))
    print("\nreq/s climbs because vLLM batches concurrent requests into shared GPU passes.")
else:
    print("Skipped.")


  concurrency= 1  wall= 0.42s   2.39 req/s
  concurrency= 2  wall= 0.50s   3.97 req/s
  concurrency= 4  wall= 0.57s   6.98 req/s
  concurrency= 8  wall= 0.62s  12.98 req/s
  concurrency=16  wall= 0.73s  21.96 req/s

 concurrency  wall_s  req_per_s
           1    0.42       2.39
           2    0.50       3.97
           4    0.57       6.98
           8    0.62      12.98
          16    0.73      21.96

req/s climbs because vLLM batches concurrent requests into shared GPU passes.


## 9. Four production metrics + cost

Cost uses the price of whichever instance the sweep selected.

In [10]:
# Block 9 - Production metrics (streaming) + cost
import numpy as np, pandas as pd

def measure(n_requests=20, max_new_tokens=64):
    prompt = "Write two sentences about why LLM inference is memory bound:"
    ttfts, tpots, totals, counts = [], [], [], []
    wall0 = time.perf_counter()
    for _ in range(n_requests):
        t0 = time.perf_counter()
        stream = invoke_vllm(prompt, max_new_tokens=max_new_tokens, stream=True)
        first_t = None; n = 0
        for event in stream:
            chunk = event.get("PayloadPart", {}).get("Bytes", b"")
            if not chunk: continue
            if first_t is None: first_t = time.perf_counter()
            n += 1
        t_end = time.perf_counter()
        if first_t is None: first_t = t_end
        ttfts.append((first_t-t0)*1000)
        tpots.append(((t_end-first_t)/max(n-1,1))*1000)
        totals.append((t_end-t0)*1000); counts.append(n)
    wall = time.perf_counter()-wall0
    return {
        "TTFT_p50_ms":      round(float(np.percentile(ttfts,50)),1),
        "TPOT_p50_ms":      round(float(np.percentile(tpots,50)),1),
        "latency_p99_ms":   round(float(np.percentile(totals,99)),1),
        "throughput_tok_s": round(sum(counts)/wall,1),
        "requests":         n_requests,
    }

if ALLOW_AWS_MUTATIONS:
    print(f"Measuring on real vLLM endpoint ({chosen_instance}), 20 streamed requests...\n")
    metrics = measure(20)
    dfm = pd.DataFrame([metrics]).T; dfm.columns = ["value"]
    print(dfm.to_string())
    hourly = INSTANCE_HOURLY.get(chosen_instance, 1.0)
    tok_per_hr = metrics["throughput_tok_s"] * 3600
    cost_1m = hourly / max(tok_per_hr,1) * 1_000_000
    print(f"\nInstance: {chosen_instance} (~${hourly}/hr)")
    print(f"Approx cost per 1M output tokens (streaming): ${cost_1m:.2f}")
    print("Concurrent throughput (Block 8) is higher, so real cost/token is lower.")
else:
    print("Skipped.")


Measuring on real vLLM endpoint (ml.g4dn.xlarge), 20 streamed requests...

                  value
TTFT_p50_ms        54.7
TPOT_p50_ms        10.4
latency_p99_ms    822.7
throughput_tok_s   89.4
requests           20.0

Instance: ml.g4dn.xlarge (~$0.736/hr)
Approx cost per 1M output tokens (streaming): $2.29
Concurrent throughput (Block 8) is higher, so real cost/token is lower.


## 10. Teardown — stop GPU billing

In [12]:
# Block 10 - Guarded teardown
CONFIRM_TEARDOWN = True   # <-- set True to delete endpoint + config + model

if not CONFIRM_TEARDOWN:
    print("CONFIRM_TEARDOWN is False. Set True and re-run to delete the GPU endpoint.")
    print(f"Endpoint {endpoint_name} bills while it exists - don't leave it running.")
else:
    for fn, kw, lbl in [
        (sm.delete_endpoint,        {"EndpointName": endpoint_name},        f"endpoint {endpoint_name}"),
        (sm.delete_endpoint_config, {"EndpointConfigName": endpoint_config},f"config {endpoint_config}"),
        (sm.delete_model,           {"ModelName": model_name},              f"model {model_name}"),
    ]:
        try: fn(**kw); print(f"  DELETE {lbl}")
        except ClientError as e: print(f"  skip {lbl}: {e.response['Error']['Code']}")
    print("\nTeardown complete. GPU billing stops within ~1 min.")


  DELETE endpoint lesson7-vllm-demo-ep
  DELETE config lesson7-vllm-demo-cfg-ml-g4dn-xlarge-20260816-112835
  DELETE model lesson7-vllm-demo-model-20260816-112833

Teardown complete. GPU billing stops within ~1 min.


## 11. Recap

You deployed **vLLM as a real managed AWS service** — no simulation:

1. AWS LMI container running the **vLLM engine** (PagedAttention + continuous batching + Flash Attention).
2. A real SageMaker GPU endpoint — **auto-selected** the first instance type with quota.
3. Real inference over SageMaker Runtime (single, streaming, concurrent).
4. Concurrency sweep showing continuous batching hold throughput.
5. Four production metrics + cost per 1M tokens on the actual instance used.

### Next
**Lesson 8 — Distributed Training & Scaling**: real SageMaker distributed FSDP training.